# 01 — Exploratory Data Analysis

Project #23 — Guardrails & Prompt Injection Defense.

Inspect the 30k-prompt corpus before training. Run `python -m prompt_guard.data` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROC = Path('../data/processed')
sns.set_theme(style='whitegrid')

In [ ]:
prompts = pd.read_parquet(PROC / 'prompts.parquet')
redteam = pd.read_parquet(PROC / 'redteam.parquet')
outputs = pd.read_parquet(PROC / 'outputs.parquet')
print('prompts:', len(prompts), '  redteam:', len(redteam), '  outputs:', len(outputs))
prompts.head()

## Schema and missingness

In [ ]:
print(prompts.dtypes)
print('\nMissingness:')
print(prompts.isna().sum())

## Class balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
prompts['is_injection'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'])
axes[0].set_title('Benign vs injection (full corpus)')
prompts[prompts['is_injection']==1]['injection_type'].value_counts().plot(
    kind='bar', ax=axes[1], color='tomato'
)
axes[1].set_title('Injection-type breakdown')
plt.tight_layout(); plt.show()

## Severity distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7,3))
sns.countplot(data=prompts, x='severity', order=['none','low','med','high'], ax=ax)
ax.set_title('Severity distribution')
plt.show()

## Token-length distribution by class

In [ ]:
prompts['n_tokens'] = prompts['text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(7,3))
sns.boxplot(data=prompts, x='injection_type', y='n_tokens', ax=ax)
plt.xticks(rotation=20)
ax.set_title('Token length by injection type')
plt.show()

## Vocabulary-overlap heatmap (TF-IDF means by injection type)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(max_features=400, min_df=10)
X = vec.fit_transform(prompts['text'])
df_tfidf = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
df_tfidf['injection_type'] = prompts['injection_type'].values
means = df_tfidf.groupby('injection_type').mean()
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(means.T.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Per-type TF-IDF mean correlation')
plt.show()

## Target-leakage check
Confirm that no benign prompts contain injection-pattern tokens — otherwise the classifier has a trivial shortcut.

In [ ]:
leak = prompts[prompts['is_injection']==0]['text'].str.contains('ignore previous', case=False).sum()
print('benign prompts containing "ignore previous":', int(leak))

## Takeaways
- Class balance is intentional ~17% injection.
- Each injection type has its own characteristic vocabulary.
- No benign prompts contain the giveaway phrases — clean train signal.